# 💧 LFM3-Audio - Speech Capabilities

This notebook runs the examples from the [Audio Capabilities](https://docs.liquid.ai/lfm/key-concepts/audio-capabilities), [Speech Generation](https://docs.liquid.ai/lfm/key-concepts/speech-generation), [Tool Calling](https://docs.liquid.ai/lfm/key-concepts/tool-use) documentation with [`LiquidAI/LFM3-Audio-2B`](https://huggingface.co/LiquidAI/LFM3-Audio-2B). The code matches the documentation, so you can run each example end to end.

## Setup

Install Transformers and the packages used by the examples. `librosa` and `soundfile` load and save audio files, and `datasets` loads a sample recording. A GPU is recommended; the model runs in bfloat16.

In [ ]:
!pip install "transformers>=5.20.2" torch torchvision pillow accelerate datasets librosa soundfile

Load the model and processor with `trust_remote_code=True`. The processor bundles the audio codec (`processor.audio_tokenizer`), which turns generated speech codes into a 24 kHz waveform.

In [1]:
import torch
import soundfile as sf
from transformers import AutoModelForMultimodalLM, AutoProcessor

model_id = "LiquidAI/LFM3-Audio-2B"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

model = AutoModelForMultimodalLM.from_pretrained(
    model_id, dtype=dtype, device_map=device, trust_remote_code=True
).eval()
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
codec = processor.audio_tokenizer.to(device=device).eval()

Loading weights: 100%|##########| 527/527 [00:01<00:00, 328.97it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100%|##########| 203/203 [00:00<00:00, 3455.21it/s]


Every task is a chat conversation, and the system prompt describes the task. Answers are in text by default. Some examples first record a spoken input with [text-to-speech](https://docs.liquid.ai/lfm/key-concepts/speech-generation#text-to-speech). Where the code differs between audio and omni models, the examples have a tab for each.

## Speech recognition

Use the system prompt `Perform ASR in <language>.` with the English name of the language. Pass audio in the user turn as `{"type": "audio", "audio": ...}`, where the value is a file path or URL. The processor resamples it to 16 kHz mono.

This example transcribes a LibriSpeech clip:

In [2]:
import io
from datasets import Audio as AudioFeature, load_dataset

librispeech = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
librispeech = librispeech.cast_column("audio", AudioFeature(decode=False))

sample = librispeech[0]
speech, sr = sf.read(io.BytesIO(sample["audio"]["bytes"]))
sf.write("librispeech.wav", speech, sr)
print("Reference:", sample["text"])

messages = [
    {"role": "system", "content": "Perform ASR in English."},
    {"role": "user", "content": [{"type": "audio", "audio": "librispeech.wav"}]},
]

# Apply chat template
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=512, suppress_mixed=True, suppress_audio_out=True)

transcript = processor.decode(outputs.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Transcript:", transcript)

Reference: MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL
Transcript: Mr. Quilter is the apostle of the middle classes, and we are glad to welcome his gospel.


In [3]:
from IPython.display import Audio, display

display(Audio("librispeech.wav"))

## Live speech recognition

Live speech recognition transcribes speech while it is still being spoken, instead of waiting for the full recording. Use the system prompt `Perform live ASR.`. Check the model page for supported languages.

The model processes audio in short chunks and writes the text for each chunk as soon as the chunk arrives. `model.transcribe_live()` runs one chunk at a time and returns a state that you pass to the next call. In an application, you open a microphone stream into the model and a text stream out of it. This example simulates the microphone by feeding the LibriSpeech clip in 100 ms pieces:

In [4]:
import librosa

tokenizer = processor.tokenizer
chunk_frames = 8 * model.config.interleaved_n_audio
prompt = tokenizer(
    "<|startoftext|><|im_start|>system\nPerform live ASR.<|im_end|>\n<|im_start|>assistant\n",
    add_special_tokens=False,
    return_tensors="pt",
).input_ids.to(device)
suppress = tokenizer.convert_tokens_to_ids(["<|mixed_end|>", "<|endoftext|>"])
stop = tokenizer.convert_tokens_to_ids(["<|endoftext|>", "<|im_start|>"])

state, feature_cache = None, None
mel = torch.empty((1, processor.feature_extractor.feature_size, 0), device=device)


def transcribe(frames, is_final=False):
    global state
    output = model.transcribe_live(
        frames,
        state,
        input_ids=prompt if state is None else None,
        is_final=is_final,
        max_new_tokens=96 if is_final else 48,
        suppress_token_ids=suppress,
        stop_token_ids=stop,
    )
    state = output.state
    print(tokenizer.decode(output.sequences[0], skip_special_tokens=True), end="", flush=True)


waveform, _ = librosa.load("librispeech.wav", sr=16_000)
for start in range(0, len(waveform), 1_600):  # 100 ms microphone chunks
    features, feature_cache = processor.feature_extractor.stream(
        waveform[start:start + 1_600], feature_cache
    )
    mel = torch.cat((mel, features.input_features.to(device)), dim=-1)
    while mel.shape[-1] >= chunk_frames:
        transcribe(mel[..., :chunk_frames])
        mel = mel[..., chunk_frames:]

# Flush the remaining audio and end the transcript
features, feature_cache = processor.feature_extractor.stream([], feature_cache, is_final=True)
mel = torch.cat((mel, features.input_features.to(device)), dim=-1)
while mel.shape[-1] > chunk_frames:
    transcribe(mel[..., :chunk_frames])
    mel = mel[..., chunk_frames:]
transcribe(mel, is_final=True)
print()

Mr. Quilter is the apostle of the middle classes, and we're glad to welcome his gospel.


For streams longer than 30 seconds, pass `input_ids=prompt` again to start a new text context. The encoder keeps following the audio stream.

## Speech translation

Models with speech translation support can translate speech into text. Name the source and target languages in the system prompt, for example `Convert the speech in {source} to written {target}.` or `Translate the following speech from {source} to {target} in text form.` Try the phrasing that works best for your data.

For text-to-text translation, use a [text model](https://docs.liquid.ai/lfm/models/text-models) instead.

This example first records a French sentence with text-to-speech:

In [5]:
tts_messages = [
    {"role": "system", "content": "Perform TTS in French using a male voice."},
    {"role": "user", "content": "On pourrait inventer une nouvelle recette ce soir. Si le résultat est étrange, on lui donnera un nom très chic."},
]

In [6]:
# Apply chat template
inputs = processor.apply_chat_template(
    tts_messages + [{"role": "assistant", "content": "<|audio_start|>"}],
    continue_final_message=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

torch.manual_seed(0)
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        audio_temperature=1.0,
        audio_top_k=128,
        audio_repetition_penalty=1.1,
        audio_repetition_window=4096,
    )

codes = outputs.audio_codes[:1, :, : outputs.audio_lengths[0]]
with torch.inference_mode():
    speech = codec.decode(codes).audio_values[0].float().cpu().numpy().squeeze()
sf.write("tts_fr.wav", speech, 24_000)

In [7]:
from IPython.display import Audio, display

display(Audio("tts_fr.wav"))

Then translate the recording into English:

In [8]:
messages = [
    {"role": "system", "content": "Convert the speech in French to written English."},
    {"role": "user", "content": [{"type": "audio", "audio": "tts_fr.wav"}]},
]

# Apply chat template
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=512, suppress_mixed=True, suppress_audio_out=True)

translation = processor.decode(outputs.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Translation:", translation)

Translation: We could invent a new recipe this evening. If the result is strange, we will give it a very fancy name.


## Speech-to-text chat

In this mode, the model behaves like a text LLM that also accepts speech. No system prompt is needed.

This example first records a spoken question with text-to-speech, then asks it:

In [9]:
tts_messages = [
    {"role": "system", "content": "Perform TTS in American English using a male voice."},
    {"role": "user", "content": "I want to move more, but I have not found an activity I enjoy. How can I try a few options and build a small habit?"},
]
messages = [
    {"role": "user", "content": [{"type": "audio", "audio": "question.wav"}]},
]

Record the question:

In [10]:
# Apply chat template
inputs = processor.apply_chat_template(
    tts_messages + [{"role": "assistant", "content": "<|audio_start|>"}],
    continue_final_message=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

torch.manual_seed(0)
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        audio_temperature=1.0,
        audio_top_k=128,
        audio_repetition_penalty=1.1,
        audio_repetition_window=4096,
    )

codes = outputs.audio_codes[:1, :, : outputs.audio_lengths[0]]
with torch.inference_mode():
    speech = codec.decode(codes).audio_values[0].float().cpu().numpy().squeeze()
sf.write("question.wav", speech, 24_000)

In [11]:
from IPython.display import Audio, display

display(Audio("question.wav"))

Prefill `<think>` to let the model reason first. It closes the reasoning with `</think>`, and the answer follows. Pass `suppress_mixed=True` and `suppress_audio_out=True` to keep the answer in text:

In [12]:
# Apply chat template
inputs = processor.apply_chat_template(
    messages + [{"role": "assistant", "content": "<think>"}],
    continue_final_message=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=2048, suppress_mixed=True, suppress_audio_out=True)

text = processor.decode(outputs.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
thinking, separator, answer = text.partition("</think>")
if not separator or not answer.strip():
    raise RuntimeError("No final answer was generated; increase max_new_tokens.")
print("Thinking:", thinking.strip())
print("Answer:", answer.strip())

Thinking: We need to answer the question: "I want to move more, but I have not found an activity I enjoy. How can I try a few options and build a small habit?"

1.  **Analyze the User's Request:**
    *   **Goal:** Move more.
    *   **Problem:** Hasn't found an activity they enjoy.
    *   **Solution Needed:** How to try a few options and build a small habit.

2.  **Deconstruct the Problem:**
    *   *Why is it hard to move?* Lack of enjoyment, lack of motivation, lack of structure, or simply not knowing what to do.
    *   *How to solve it?* Experimentation, habit formation, small steps, and finding intrinsic motivation.

3.  **Brainstorming Strategies (Categorization for Clarity):**
    *   *Phase 1: Exploration (Trying options)* - How to find the right activity without feeling overwhelmed.
    *   *Phase 2: Habit Formation (Building the routine)* - How to turn a new activity into a daily habit.
    *   *Phase 3: Motivation & Mindset* - How to keep going when it's hard.

4.  **Flesh

To skip thinking, prefill an empty `<think></think>` block instead, so the answer starts immediately.

## Audio understanding

Text and audio can be mixed in the same user turn, for example a recording plus a written question about it. Questions can address speech, music, background sounds, or events.

This example asks a question about the LibriSpeech clip from [Speech recognition](https://docs.liquid.ai/lfm/key-concepts/audio-capabilities#speech-recognition):

In [13]:
messages = [{
    "role": "user",
    "content": [
        {"type": "audio", "audio": "librispeech.wav"},
        {"type": "text", "text": "Who or what is the speaker talking about, and what is their attitude towards it?"},
    ],
}]

In [14]:
# Apply chat template
inputs = processor.apply_chat_template(
    messages + [{"role": "assistant", "content": "<think></think>"}],
    continue_final_message=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=512, suppress_mixed=True, suppress_audio_out=True)

answer = processor.decode(outputs.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Answer:", answer.strip())

Answer: The speaker is talking about **Mr. Quilter**, who is described as “the apostle of the middle classes.” The speaker’s attitude is one of welcome and approval—“we’re glad to welcome his gospel,” indicating a positive, supportive stance toward Mr. Quilter’s message.


For multiple choice, use the system prompt `Answer questions about the provided audio clip. Base every answer only on what can be heard.` and end the user turn with `Please choose the answer from the options.`

In [15]:
messages = [
    {"role": "system", "content": "Answer questions about the provided audio clip. Base every answer only on what can be heard."},
    {"role": "user", "content": [
        {"type": "audio", "audio": "librispeech.wav"},
        {"type": "text", "text": "What does the speaker welcome?\nA. A new law\nB. Mr. Quilter's gospel\nC. A visiting choir\nD. The end of the middle classes\nPlease choose the answer from the options."},
    ]},
]

# Apply chat template
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=512, suppress_mixed=True, suppress_audio_out=True)

answer = processor.decode(outputs.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(answer)

Mr. Quilter's gospel


## Choose the output modality

Generation starts in text mode and switches to speech or interleaved output when the model emits an output-mode token. To select a mode from the start, prefill the assistant turn: add an assistant message with one of the prefixes below and pass `continue_final_message=True` to `apply_chat_template()`.

| Assistant prefill | Behavior |
|---|---|
| None | Start in text mode. The model can still emit a token that switches modes. |
| `<\|audio_start\|>` | Start speech-only output, for example text-to-speech, until an end-of-audio frame. |
| `<\|mixed_start\|>` | Start interleaved text and speech, alternating blocks of text tokens and speech frames. |
| `<think>` | Begin a thinking block. The answer follows the closing `</think>`. |
| `<think></think>` | Supply an empty thinking block, so the answer starts immediately. |

The thinking prefixes are for text-only answers. For those, also pass `suppress_mixed=True` and `suppress_audio_out=True` to `model.generate()`, so the model doesn't switch to speech.

## Text-to-speech

Use the system prompt `Perform TTS in <language> using a <female|male> voice.` For English, say `American English` or `British English`.

Check the model page for supported languages and voices.

This example speaks a sentence in American English:

In [16]:
messages = [
    {"role": "system", "content": "Perform TTS in American English using a female voice."},
    {"role": "user", "content": "I made a little plan for today: go for a walk, find a new workout playlist, and try a new cake recipe."},
]

Prefill `<|audio_start|>`:

In [17]:
# Apply chat template
inputs = processor.apply_chat_template(
    messages + [{"role": "assistant", "content": "<|audio_start|>"}],
    continue_final_message=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

torch.manual_seed(0)
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        audio_temperature=1.0,
        audio_top_k=128,
        audio_repetition_penalty=1.1,
        audio_repetition_window=4096,
    )

codes = outputs.audio_codes[:1, :, : outputs.audio_lengths[0]]
with torch.inference_mode():
    speech = codec.decode(codes).audio_values[0].float().cpu().numpy().squeeze()
sf.write("tts_en.wav", speech, 24_000)

In [18]:
from IPython.display import Audio, display

display(Audio("tts_en.wav"))

To check the result, pass the file to the [speech recognition](https://docs.liquid.ai/lfm/key-concepts/audio-capabilities#speech-recognition) example.

## Spoken chat

Use the system prompt `Respond with interleaved text and audio.` The model answers conversationally in text and speech at the same time, so speech can start playing before the full answer is written. Keep the system prompt exactly as shown, and give any further instructions in the user turn.

This example first records a spoken question with text-to-speech, then asks it and saves the spoken reply to `reply.wav`:

In [19]:
tts_messages = [
    {"role": "system", "content": "Perform TTS in American English using a male voice."},
    {"role": "user", "content": "I want to move more, but I have not found an activity I enjoy. How can I try a few options and build a small habit?"},
]
messages = [
    {"role": "system", "content": "Respond with interleaved text and audio."},
    {"role": "user", "content": [{"type": "audio", "audio": "question.wav"}]},
]

Record the question:

In [20]:
# Apply chat template
inputs = processor.apply_chat_template(
    tts_messages + [{"role": "assistant", "content": "<|audio_start|>"}],
    continue_final_message=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

torch.manual_seed(0)
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        audio_temperature=1.0,
        audio_top_k=128,
        audio_repetition_penalty=1.1,
        audio_repetition_window=4096,
    )

codes = outputs.audio_codes[:1, :, : outputs.audio_lengths[0]]
with torch.inference_mode():
    speech = codec.decode(codes).audio_values[0].float().cpu().numpy().squeeze()
sf.write("question.wav", speech, 24_000)

In [21]:
from IPython.display import Audio, display

display(Audio("question.wav"))

Then ask it. Prefill `<|mixed_start|>`:

In [22]:
# Apply chat template
inputs = processor.apply_chat_template(
    messages + [{"role": "assistant", "content": "<|mixed_start|>"}],
    continue_final_message=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
        audio_temperature=1.0,
        audio_top_k=128,
        audio_repetition_penalty=1.1,
        audio_repetition_window=4096,
    )

reply = processor.decode(outputs.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Text:", reply)

codes = outputs.audio_codes[:1, :, : outputs.audio_lengths[0]]
with torch.inference_mode():
    speech = codec.decode(codes).audio_values[0].float().cpu().numpy().squeeze()
sf.write("reply.wav", speech, 24_000)

Text: Try picking three low‑commitment activities you like, like a short walk, a cooking class, or a beginner yoga session, and schedule one each week for a month to see what clicks. Keep a simple log of how you felt and note any patterns that make you look forward to it. Does that sound doable for you?


In [23]:
from IPython.display import Audio, display

display(Audio("reply.wav"))

## Stream speech

For streaming applications, `model.generate()` accepts `streamer` for text tokens and `audio_streamer` for speech frames. The codec decodes those frames incrementally, so speech can be played as it is generated.

## Tool calling

Audio and omni models with tool-calling support can take spoken requests and return tool calls. This example turns one spoken request into two function calls: cancel a streaming-platform subscription at the end of the billing period, then subscribe to a podcast platform's Max plan with monthly billing. StreamBox and PodSpace are fictional services, and the example only prints the proposed calls.

This example shows how to define the tools and generate the tool calls. Executing the calls and generating the final answer work the same as in [Text Models](https://docs.liquid.ai/lfm/key-concepts/tool-use#text-models).

The example first records the spoken request with [text-to-speech](https://docs.liquid.ai/lfm/key-concepts/speech-generation#text-to-speech). Define the request, the tools, and the instructions:

In [24]:
import json

tools = [
    {
        "name": "cancel_subscription",
        "description": "Cancel the signed-in user's subscription to a streaming platform at the specified time.",
        "parameters": {
            "type": "object",
            "properties": {
                "platform": {"type": "string", "description": "Name of the streaming platform, for example StreamBox."},
                "effective_when": {
                    "type": "string",
                    "enum": ["immediately", "end_of_billing_period"],
                },
            },
            "required": ["platform", "effective_when"],
        },
    },
    {
        "name": "subscribe_to_podcast_platform",
        "description": "Subscribe the signed-in user to a podcast platform with the chosen plan and billing frequency.",
        "parameters": {
            "type": "object",
            "properties": {
                "platform": {"type": "string", "description": "Name of the podcast platform, for example PodSpace."},
                "plan": {"type": "string", "enum": ["basic", "plus", "max"]},
                "billing_frequency": {"type": "string", "enum": ["monthly", "annual"]},
            },
            "required": ["platform", "plan", "billing_frequency"],
        },
    },
]
instructions = (
    "Return tool calls for both requested actions, in the order requested. "
    "Do not claim that an action has completed without a tool result."
)

tts_messages = [
    {"role": "system", "content": "Perform TTS in British English using a female voice."},
    {"role": "user", "content": "Please cancel my StreamBox subscription at the end of the current billing period. Then subscribe me to PodSpace on the Max plan, billed monthly."},
]

Record the request:

In [25]:
# Apply chat template
inputs = processor.apply_chat_template(
    tts_messages + [{"role": "assistant", "content": "<|audio_start|>"}],
    continue_final_message=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

torch.manual_seed(0)
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        audio_temperature=1.0,
        audio_top_k=128,
        audio_repetition_penalty=1.1,
        audio_repetition_window=4096,
    )

codes = outputs.audio_codes[:1, :, : outputs.audio_lengths[0]]
with torch.inference_mode():
    speech = codec.decode(codes).audio_values[0].float().cpu().numpy().squeeze()
sf.write("request.wav", speech, 24_000)

In [26]:
from IPython.display import Audio, display

display(Audio("request.wav"))

List the tools in the system prompt, between `<tools>` and `</tools>`, one JSON object per line:

In [27]:
messages = [
    {
        "role": "system",
        "content": instructions + "\nList of tools:\n<tools>\n" + "\n".join(json.dumps(t) for t in tools) + "\n</tools>",
    },
    {"role": "user", "content": [{"type": "audio", "audio": "request.wav"}]},
]

# Apply chat template
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=512, suppress_mixed=True, suppress_audio_out=True)

call = processor.decode(outputs.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(call)

<tool_call>
<function=cancel_subscription>
<parameter=platform>
StreamBox
</parameter>
<parameter=effective_when>
end_of_billing_period
</parameter>
</function>
</tool_call>
<tool_call>
<function=subscribe_to_podcast_platform>
<parameter=platform>
PodSpace
</parameter>
<parameter=plan>
max
</parameter>
<parameter=billing_frequency>
monthly
</parameter>
</function>
</tool_call>


Don't prefill a thinking block for tool calls; the model calls tools without reasoning first.

To get a spoken answer once the tool results are in, start the system prompt with `Respond with function calls, and interleaved text and audio.\n\n`, followed by any instructions and the tool list, and don't pass `suppress_mixed` or `suppress_audio_out`. The model then either calls tools or answers in interleaved text and speech. The model card's [tool calling example](https://huggingface.co/LiquidAI/LFM3-Audio-2B) shows the full loop, including returning tool results.

# 🚀 Next steps

- Fine-tune LFM3-Audio on your own data: see [`lfm3_audio_tts_sft_with_transformers.ipynb`](../finetuning/notebooks/lfm3_audio_tts_sft_with_transformers.ipynb), which teaches the model a new voice.
- Explore LFM3-Omni, which adds image and video understanding on top of everything above: [`lfm3_omni_capabilities.ipynb`](lfm3_omni_capabilities.ipynb).
- Documentation: [docs.liquid.ai](https://docs.liquid.ai/lfm) · Community: [Discord](https://discord.com/invite/liquid-ai)